In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ['GROQ_API_KEY']:
    print("API key is set.")
    

API key is set.


In [2]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant", 
    temperature=0,
    
)

d:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


EXTRACTING TEXTS FROM PDFS


In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader=PyPDFLoader("./Docs/fabric onelake.pdf")

docs=loader.load()
docs


[Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2025-06-04T06:50:30+02:00', 'msip_label_87ba5c36-b7cf-4793-bbc2-bd5b3a9f95ca_siteid': '72f988bf-86f1-41af-91ab-2d7cd011db47', 'msip_label_87ba5c36-b7cf-4793-bbc2-bd5b3a9f95ca_method': 'Privileged', 'msip_label_87ba5c36-b7cf-4793-bbc2-bd5b3a9f95ca_enabled': 'True', 'title': 'PowerPoint Presentation', 'author': 'Aaron Merrill', 'moddate': '2025-06-04T06:50:30+02:00', 'source': './Docs/fabric onelake.pdf', 'total_pages': 61, 'page': 0, 'page_label': '1'}, page_content='Secure data end-to-end with \nMicrosoft Fabric and OneLake\nAaron Merrill\nPrincipal Product Manager\nMicrosoft\n@linkedin.com/in/aajmerrill/'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2025-06-04T06:50:30+02:00', 'msip_label_87ba5c36-b7cf-4793-bbc2-bd5b3a9f95ca_siteid': '7

Changing metadata

In [4]:
for i in docs:
    i.metadata={"source":"fabric onelake.pdf","developer":"Microsoft"}

In [5]:
docs[0].metadata

{'source': 'fabric onelake.pdf', 'developer': 'Microsoft'}

In [6]:
#step 2: Chunking

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter= RecursiveCharacterTextSplitter(
chunk_size=1000,
chunk_overlap=100)

chunks=splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'fabric onelake.pdf', 'developer': 'Microsoft'}, page_content='Secure data end-to-end with \nMicrosoft Fabric and OneLake\nAaron Merrill\nPrincipal Product Manager\nMicrosoft\n@linkedin.com/in/aajmerrill/'),
 Document(metadata={'source': 'fabric onelake.pdf', 'developer': 'Microsoft'}, page_content='Agenda\nOneLake and security in \nMicrosoft Fabric\n1\nSecurity model details2\nAPI details3\nShortcuts and integration \npatterns\n4\nManaging OneLake security5\nWrap up6'),
 Document(metadata={'source': 'fabric onelake.pdf', 'developer': 'Microsoft'}, page_content='OneLake and security in Microsoft Fabric'),
 Document(metadata={'source': 'fabric onelake.pdf', 'developer': 'Microsoft'}, page_content='OneLake for all data\n“The OneDrive for data”\n OneDrive\nfor documents\nOneLake\nfor data\nOneLake provides a data lake as a service \nwithout you needing to build it'),
 Document(metadata={'source': 'fabric onelake.pdf', 'developer': 'Microsoft'}, page_content='

In [7]:
chunks[0].metadata

{'source': 'fabric onelake.pdf', 'developer': 'Microsoft'}

In [8]:
#embeddings

from langchain_huggingface import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2913.22it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
#vectorstores

from langchain_community.vectorstores import Chroma
vectorstores=Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,

)

In [10]:
#semantic search
#semantic search
vectorstores.similarity_search("Why do we need one lake",k=3)


[Document(metadata={'developer': 'Microsoft', 'source': 'fabric onelake.pdf'}, page_content='OneLake for all data\n“The OneDrive for data”\n OneDrive\nfor documents\nOneLake\nfor data\nOneLake provides a data lake as a service \nwithout you needing to build it'),
 Document(metadata={'source': 'fabric onelake.pdf', 'developer': 'Microsoft'}, page_content='OneLake security feature - now\n• Lakehouse\n• Azure Mirrored Databrick Catalog\nDefine OneLake security on:\n OneLake security filtering:\n• Lakehouse\n• Spark notebooks\n• SQL Analytics Endpoint\n• Semantic Models'),
 Document(metadata={'source': 'fabric onelake.pdf', 'developer': 'Microsoft'}, page_content='One Copy, many security definitions\nEnabling fine-grained security at each engine\nEach engine in Fabric provides its \nown robust set of security \ncapabilities.\nServerless\nCompute\nDelta – Parquet \nFormat\nDelta – Parquet \nFormat\nDelta – Parquet \nFormat\nDelta – Parquet \nFormat\nT-SQL\nSpark\n KQL\n Analysis\nServices\n

Talk to llm